Ideas for the data analysis component:
- overview:
    1. number of transactions
    2. number of features -> make aggr. / non-aggr. distinction
    3. number of edges
    4. number of time slices
    
- subset statistics (whole, split, slice), feature-based:
    1. number of transactions with label distribution
    2. feature distributions: means, min/max, quantiles, standard deviation (will reveal normalization leakage)
    3. features correlation matrix
    4. PCA explained variation
    
- subset statistics (whole, split, slice), graph-based (whole):
    1. number of considered edges
    2. in-degree/out-degree distributions (means, min/max, quantiles, std. deviation)
    3. clustering coefficient 
    4. distribution of subgraph sizes (means, min/max, quantiles, std. deviation)

- subset statistics (whole, split, slice), graph-based (labeled-only):
    1. number of considered edges
    2. in-degree/out-degree distributions (means, min/max, quantiles, std. deviation)
    3. clustering coefficient 
    4. distribution of subgraph sizes (means, min/max, quantiles, std. deviation)
    
        
Functionality:
- loads data from data/raw directory
- splits data per slice
- splits data per train-val-test split
- computes statistics

Limitations fount along the way:
- features that are normalized across all slices already contain future leakage
    -> features that are based on the out-degree (not transaction outputs!) of a transaction contain future information
- the graph within an unfiltered slice is one weakly connected component
- clustering coefficient in acyclic graphs is 0 (e.g. no triangles possible)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

## 1. Load data

In [ ]:
raw_path =  "../data/raw/"
features_path = raw_path + "elliptic_txs_features.csv"
classes_path = raw_path + "elliptic_txs_classes.csv"
edges_path = raw_path + "elliptic_txs_edgelist.csv"

def load_data(f_path, c_path, e_path):
    # features
    txs_f_df = pd.read_csv(f_path, header=None)
    txs_f_df.columns = (["txId", "slice"] 
                        + ["f" + str(i+1) for i in range(len(txs_f_df.columns)-2)])
    # classes
    txs_c_df = pd.read_csv(c_path)
    # edges
    txs_e_df = pd.read_csv(e_path)
    return txs_f_df, txs_c_df, txs_e_df

In [ ]:
features_df, classes_df, edgelist_df = load_data(features_path, classes_path, edges_path)
print(f"{features_df.shape=}, {classes_df.shape=}, {edgelist_df.shape=}")

## 2. Split data into slices and train-val-test sets

In [ ]:
# first, split the features set into slices, since the slice is directly given in the columns
def get_features_slices() -> list[pd.DataFrame]:
    grouped_df = features_df.groupby("slice")
    f_sls = []
    for s in features_df["slice"].unique():
        f_sls.append(grouped_df.get_group(s))
    return f_sls
        
f_slices_list = get_features_slices()
print(f"{len(f_slices_list)=}")


In [ ]:
# second, split the labels set into slices
# slice column is added to classes df
def get_classes_slices() -> list[pd.DataFrame]:
    c_sls = []
    for f_slice in f_slices_list:
        f_slice = f_slice[["txId", "slice"]]
        inner_df = pd.merge(classes_df, f_slice, on="txId", how="inner")
        c_slice_df = inner_df[inner_df["slice"] == f_slice["slice"].iloc[0]]
        # c_slice_df = c_slice_df.drop("slice", axis=1)
        c_sls.append(c_slice_df)
    return c_sls
    
c_slices_list = get_classes_slices()
print(f"{len(c_slices_list)=}")
print(c_slices_list[0].head())

In [ ]:
# third, split the edges into slices
# condition: f.txId == e.txId1 and f.txId == e.txId2
# slice column is added to classes df
def get_edge_slices() -> list[pd.DataFrame]:
    e_sls = []
    for f_slice in f_slices_list:
        f_slice = f_slice[["txId", "slice"]]
        inner_df_1 = pd.merge(edgelist_df, f_slice, left_on="txId1", right_on="txId", how="inner")
        c_slice_df_1 = inner_df_1[inner_df_1["slice"] == f_slice["slice"].iloc[0]]
        c_slice_df_1.drop(["txId"], axis=1, inplace=True)
        
        inner_df_2 = pd.merge(edgelist_df, f_slice, left_on="txId2", right_on="txId", how="inner")
        c_slice_df_2 = inner_df_2[inner_df_2["slice"] == f_slice["slice"].iloc[0]]
        c_slice_df_2.drop(["txId"], axis=1, inplace=True)
        
        # note that self-edges are inadmissible by Bitcoin's UTXO protocol
        c_slice_df = pd.concat([c_slice_df_1, c_slice_df_2])
        e_sls.append(c_slice_df)
    return e_sls

e_slices_list = get_edge_slices()
print(f"{len(e_slices_list)=}")
print(e_slices_list[0].head())


## 3. Split data into training, validation, and test sets
The train-val-test ratio will be fixed to 29:5:15 to keep results comparable to previous work, which usually uses a train-test ratio of 34:15, omitting the validation step.

In [ ]:
def data_splitter(slice_list, train=29, val=5, test=15):
    return slice_list[:train], slice_list[train:train+val], slice_list[train+val:train+val+test]

f_train_slices_list, f_val_slices_list, f_test_slices_list = data_splitter(f_slices_list)
c_train_slices_list, c_val_slices_list, c_test_slices_list = data_splitter(c_slices_list)
e_train_slices_list, e_val_slices_list, e_test_slices_list = data_splitter(e_slices_list)

print(f"{len(f_train_slices_list)=}")
print(f"{len(f_val_slices_list)=}")
print(f"{len(f_test_slices_list)=}\n")

print(f"{len(c_train_slices_list)=}")
print(f"{len(c_val_slices_list)=}")
print(f"{len(c_test_slices_list)=}\n")

print(f"{len(e_train_slices_list)=}")
print(f"{len(e_val_slices_list)=}")
print(f"{len(e_test_slices_list)=}")

## 4. Compute statistics
### 4.1 Overview


In [ ]:
# number of transactions
n_tx = features_df.shape[0]
# number of features (deduct 2 for txId column and slice column)
n_feats = features_df.columns.size - 2
# number of local features excluding the slice column (from Weber et al. 2019)
n_feats_local = 93
# number of aggregated features (from Weber et al. 2019)
n_feats_aggr = n_feats - n_feats_local

print(f"{n_tx=}")
print(f"{n_feats=}")
print(f"{n_feats_local=}")
print(f"{n_feats_aggr=}")

In [ ]:
# number of edges
n_edges = edgelist_df.shape[0]
# number of time slices
n_slices = len(f_slices_list)

print(f"{n_edges=}")
print(f"{n_slices=}")

### 4.2 Subset statistics (whole, split, slice), feature-based
#### 4.2.1 Label distribution

In [ ]:
# whole
n_tx_unk = classes_df[classes_df["class"] == "unknown"].shape[0]
n_tx_illicit = classes_df[classes_df["class"] == "1"].shape[0]
n_tx_licit = classes_df[classes_df["class"] == "2"].shape[0]

print(f"{n_tx_unk=}")
print(f"{n_tx_illicit=}")
print(f"{n_tx_licit=}")

In [ ]:
# per split

# training split
c_tx_train = pd.concat(c_train_slices_list)

n_tx_train = c_tx_train.shape[0]
n_tx_train_unk = c_tx_train[c_tx_train["class"] == "unknown"].shape[0]
n_tx_train_illicit = c_tx_train[c_tx_train["class"] == "1"].shape[0]
n_tx_train_licit = c_tx_train[c_tx_train["class"] == "2"].shape[0]

print(f"{n_tx_train=}")
print(f"{n_tx_train_unk=}")
print(f"{n_tx_train_illicit=}")
print(f"{n_tx_train_licit=}")

In [ ]:
# validation split
c_tx_val = pd.concat(c_val_slices_list)

n_tx_val = c_tx_val.shape[0]
n_tx_val_unk = c_tx_val[c_tx_val["class"] == "unknown"].shape[0]
n_tx_val_illicit = c_tx_val[c_tx_val["class"] == "1"].shape[0]
n_tx_val_licit = c_tx_val[c_tx_val["class"] == "2"].shape[0]

print(f"{n_tx_val=}")
print(f"{n_tx_val_unk=}")
print(f"{n_tx_val_illicit=}")
print(f"{n_tx_val_licit=}")

In [ ]:
# test split
c_tx_test = pd.concat(c_test_slices_list)

n_tx_test = c_tx_test.shape[0]
n_tx_test_unk = c_tx_test[c_tx_test["class"] == "unknown"].shape[0]
n_tx_test_illicit = c_tx_test[c_tx_test["class"] == "1"].shape[0]
n_tx_test_licit = c_tx_test[c_tx_test["class"] == "2"].shape[0]

print(f"{n_tx_test=}")
print(f"{n_tx_test_unk=}")
print(f"{n_tx_test_illicit=}")
print(f"{n_tx_test_licit=}")

In [ ]:
# per slice
n_tx_slices = [s.shape[0] for s in c_slices_list]
n_tx_slices_unk = [s[s["class"] == "unknown"].shape[0] for s in c_slices_list]
n_tx_slices_illicit = [s[s["class"] == "1"].shape[0] for s in c_slices_list]
n_tx_slices_licit = [s[s["class"] == "2"].shape[0] for s in c_slices_list]

In [ ]:
# visualize the class distribution over all 49 slices in a bar chart

tiks = np.array(range(len(n_tx_slices))) + 1
tx_counts = np.stack((n_tx_slices_unk, n_tx_slices_illicit, n_tx_slices_licit), axis=0)
width = 0.5

fig, ax = plt.subplots()
bottom = np.zeros(len(n_tx_slices))

for l, tx_count in zip(["unknown", "illicit", "licit"], tx_counts):
    p = ax.bar(tiks, tx_count, width, label=l, bottom=bottom)
    bottom += tx_count

ax.set_title("Number of Transactions per Slice by Class")
ax.legend(loc="upper right")
ax.set_xlabel("Slice")
ax.set_ylabel("Transactions")

plt.show()

### 4.2.2 Feature Distribution

In [ ]:
# whole
feats_only_df = features_df.drop(["txId", "slice"], axis=1)

overall_stat_df = pd.DataFrame(feats_only_df.mean(axis=0), columns=["mean"])
overall_stat_df["std"] = feats_only_df.std(axis=0, ddof=0)
overall_stat_df["median"] = feats_only_df.quantile(q=0.5, axis=0)
overall_stat_df["q25"] = feats_only_df.quantile(q=0.25, axis=0)
overall_stat_df["q75"] = feats_only_df.quantile(q=0.75, axis=0)
overall_stat_df["min"] = feats_only_df.min(axis=0)
overall_stat_df["max"] = feats_only_df.max(axis=0)

print(overall_stat_df.tail())

In [ ]:
plt.boxplot(feats_only_df[["f"+str(i+161) for i in range(5)]], sym=".")

In [ ]:
# determine data range
data_min = feats_only_df["f1"].min()
data_max = feats_only_df["f1"].max()

# define equidistant bins
bins = np.arange(data_min, data_max + 2, 2)

# plot histogr.
counts, bins = np.histogram(feats_only_df["f1"], bins=bins)
plt.stairs(counts, bins)

# place number over bars
for i in range(len(counts)):
    bin_center = (bins[i] + bins[i+1]) / 2
    offset = max(counts) * 0.02
    plt.text(bin_center, counts[i] + offset, str(counts[i]), ha='center', va='bottom')

plt.show()

The distributions of the features show partially high skewness to the right side. I presume, we have underlying data that is either count-based or value-based. Both are non-negative quantities and might require special treatment for scaling or transformation (Box-Cox, Log, etc.).

Knowledge about the distribution of the features (strictly said the knowledge about the distribution of the features in the training set) can help to improve performance of classification algorithms that are not rank-preserving. Details tbd

Eventually, the knowledge about non-negativity could help to revert the already applied standard scaler.

For feature list that is not affected by the issues mentioned above, see handwritten sheet.



### 4.2.3 Feature Correlation

In [ ]:
# exemplary on the whole data set (across all 49 slices)
corr_matrix = feats_only_df.corr()
abs_corr_matrix = np.abs(corr_matrix)

mask = np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool)
# mask out all duplicates and self-pairs + represent as a list (stack)
top_pairs = abs_corr_matrix.where(mask).stack().nlargest(10)

print("Top 10 correlated pairs: \n", top_pairs)

# sorted list of features that are part of a highest corr. pair
top_features = sorted(list(set([idx for pair in top_pairs.index for idx in pair])))

plt.figure(figsize=(10, 10))
sns.heatmap(
    corr_matrix.loc[top_features, top_features], 
    annot=True, 
    cmap="coolwarm", 
    vmin=-1, 
    vmax=1,
    fmt=".2f",
    square=True,
    linewidths=0.5,
)
plt.title("Overall Feature Correlation")
plt.show()

### 4.2.4 PCA

In [ ]:
# exemplary on the whole data set (across all 49 slices)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

pca_features = pca.fit_transform(feats_only_df)
explained_variance = pca.explained_variance_ratio_

print(f"{explained_variance=}")

plt.scatter(pca_features[:100, 0], pca_features[:100, 1])
plt.show()

In [ ]:
pca_features_df = pd.DataFrame(pca_features, columns=["pc1", "pc2"])
pca_features_df["txId"] = features_df["txId"].to_numpy()

pca_features_df = pd.merge(
    pca_features_df,
    classes_df,
    on="txId",
)

# keep the first 10,000 rows.
pca_features_df = pca_features_df.head(10000).copy()

# Keep only known classes.
pca_features_df = pca_features_df[
    pca_features_df["class"].isin(["1", "2"])
].copy()

print(f"{pca_features_df.shape=}")

color_dict = {
    "1": "red",
    "2": "green",
}

fig, ax = plt.subplots()

for label in pca_features_df["class"].unique():
    mask = pca_features_df["class"].eq(label)

    ax.scatter(
        pca_features_df.loc[mask, "pc1"],
        pca_features_df.loc[mask, "pc2"],
        c=color_dict[label],
        label=label,
    )

ax.legend()
ax.set_xlabel("First Principal Component")
ax.set_ylabel("Second Principal Component")
plt.show()


### 4.3 Subset statistics (whole, split, slice), graph-based (whole)

In [ ]:
# number of edges
print(f"{n_edges=}")

In [ ]:
# statistics of in- and out-degrees

graph_out_df = pd.merge(features_df[["txId"]], edgelist_df[["txId1"]], left_on="txId", right_on="txId1", how="left").groupby(["txId"]).count()

graph_in_df = pd.merge(features_df[["txId"]], edgelist_df[["txId2"]], left_on="txId", right_on="txId2", how="left").groupby(["txId"]).count()

graph_stats_df = pd.merge(graph_in_df, graph_out_df, on="txId", how="inner")
graph_stats_df.rename({"txId1":"out", "txId2":"in"}, axis="columns", inplace=True)
print(graph_stats_df.head())
print(f"{graph_stats_df.shape=}")

In [ ]:
# out edges statistics
# mean, std., quantiles, min, max, distributions

# average in-degree 
print("mean ", graph_stats_df["in"].mean())

# standard deviation of in-degree
print("std in ", graph_stats_df["in"].std())


# 50, 25, and 75 quantiles of in-degree
print("median in ", graph_stats_df["in"].quantile(0.5))
print("25-quantile in ", graph_stats_df["in"].quantile(0.25))
print("75-quantile in ", graph_stats_df["in"].quantile(0.75))

# min and max
print("minimum in ", graph_stats_df["in"].min())
print("maximum in ", graph_stats_df["in"].max())



In [ ]:
# in edges statistics
# mean, std., quantiles, min, max, distributions
# average out-degree 
print("mean out ", graph_stats_df["out"].mean())

# standard deviation of out-degree
print("std out ", graph_stats_df["out"].std())

# 50, 25, and 75 quantiles of out-degree
print("median out ", graph_stats_df["out"].quantile(0.5))
print("25-quantile out ", graph_stats_df["out"].quantile(0.25))
print("75-quantile out ", graph_stats_df["out"].quantile(0.75))

# min and max
print("minimum out ", graph_stats_df["out"].min())
print("maximum out ", graph_stats_df["out"].max())

In [ ]:
# in-degree distribution (log-scaled y-axis)
in_stats_df = graph_stats_df.groupby(["in"]).count()
in_stats_df.rename({"out": "count"}, axis="columns", inplace=True)

print(in_stats_df.head(30))
plt.bar(in_stats_df.index, in_stats_df["count"])
plt.yscale("log")
plt.show()

In [ ]:
# out-degree distribution (log-scaled y-axis)
out_stats_df = graph_stats_df.groupby(["out"]).count()
out_stats_df.rename({"in": "count"}, axis="columns", inplace=True)

print(out_stats_df.head(30))
plt.bar(out_stats_df.index, out_stats_df["count"])
plt.yscale("log")
plt.show()

In [ ]:
# statistics about the connected components of the whole graph structure

# first, validate and clean the edge list

required_columns = {"txId1", "txId2"}

if not required_columns.issubset(edgelist_df.columns):
    missing = required_columns - set(edgelist_df.columns)
    raise ValueError(f"Missing columns: {missing}")

edges = (
    edgelist_df[["txId1", "txId2"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

print(f"Original number of edges: {len(edgelist_df):,}")
print(f"Number of cleaned unique edges: {len(edges):,}")



In [ ]:
# Second, create the directed graph, using networkx library

G = nx.from_pandas_edgelist(
    edges,
    source="txId1",
    target="txId2",
    create_using=nx.DiGraph(),
)

print(f"Number of nodes: {G.number_of_nodes():,}")
print(f"Number of edges: {G.number_of_edges():,}")

In [ ]:
# identify weakly connected components

components = list(nx.weakly_connected_components(G))

print(f"Number of connected components: {len(components):,}")

In [ ]:
# number of nodes per component
component_stats = pd.DataFrame(
    {
        "component_id": range(len(components)),
        "number_of_nodes": [len(component) for component in components],
    }
)

# number of edges per component
component_stats["number_of_edges"] = [
    G.subgraph(component).number_of_edges()
    for component in components
]

print(component_stats.head())

In [ ]:
# compute descriptive statistics

quantiles = [0.00, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]

node_size_statistics = (
    component_stats["number_of_nodes"]
    .describe(percentiles=quantiles[1:-1])
)

print("\nComponent size statistics, measured in nodes:")
print(node_size_statistics)

print("\nAdditional statistics:")
print(
    f"Mean: {component_stats['number_of_nodes'].mean():,.2f}"
)
print(
    f"Standard deviation: "
    f"{component_stats['number_of_nodes'].std():,.2f}"
)
print(
    f"Minimum: {component_stats['number_of_nodes'].min():,}"
)
print(
    f"Maximum: {component_stats['number_of_nodes'].max():,}"
)

print("\nQuantiles:")
print(
    component_stats["number_of_nodes"]
    .quantile(quantiles)
    .rename("number_of_nodes")
)

### 4.4 Subset statistics (whole, split, slice), graph-based (labeled-only)

In [ ]:
# filter all unlabeled (unknown) transactions (will reduce the edges list by a lot)
classes_filtered_df = classes_df[classes_df["class"].isin(["1", "2"])]
print(f"{classes_filtered_df.shape=}")

features_filtered_df = features_df[features_df["txId"].isin(classes_filtered_df["txId"])]
print(f"{features_filtered_df.shape=}")

edgelist_filtered_df = edgelist_df[edgelist_df["txId1"].isin(classes_filtered_df["txId"]) & edgelist_df["txId2"].isin(classes_filtered_df["txId"])]
print(f"{edgelist_filtered_df.shape=}")


In [ ]:
# statistics of in- and out-degrees

graph_out_filtered_df = pd.merge(features_filtered_df[["txId"]], edgelist_filtered_df[["txId1"]], left_on="txId", right_on="txId1", how="left").groupby(["txId"]).count()

graph_in_filtered_df = pd.merge(features_filtered_df[["txId"]], edgelist_filtered_df[["txId2"]], left_on="txId", right_on="txId2", how="left").groupby(["txId"]).count()

graph_stats_filtered_df = pd.merge(graph_in_filtered_df, graph_out_filtered_df, on="txId", how="inner")
graph_stats_filtered_df.rename({"txId1":"out", "txId2":"in"}, axis="columns", inplace=True)
print(graph_stats_filtered_df.head())
print(f"{graph_stats_filtered_df.shape=}")

In [ ]:
# out edges statistics
# mean, std., quantiles, min, max, distributions

# average in-degree 
print("mean ", graph_stats_filtered_df["in"].mean())

# standard deviation of in-degree
print("std in ", graph_stats_filtered_df["in"].std())

# 50, 25, and 75 quantiles of in-degree
print("median in ", graph_stats_filtered_df["in"].quantile(0.5))
print("25-quantile in ", graph_stats_filtered_df["in"].quantile(0.25))
print("75-quantile in ", graph_stats_filtered_df["in"].quantile(0.75))

# min and max
print("minimum in ", graph_stats_filtered_df["in"].min())
print("maximum in ", graph_stats_filtered_df["in"].max())


In [ ]:
# in edges statistics
# mean, std., quantiles, min, max, distributions
# average out-degree 
print("mean out ", graph_stats_filtered_df["out"].mean())

# standard deviation of out-degree
print("std out ", graph_stats_filtered_df["out"].std())

# 50, 25, and 75 quantiles of out-degree
print("median out ", graph_stats_filtered_df["out"].quantile(0.5))
print("25-quantile out ", graph_stats_filtered_df["out"].quantile(0.25))
print("75-quantile out ", graph_stats_filtered_df["out"].quantile(0.75))

# min and max
print("minimum out ", graph_stats_filtered_df["out"].min())
print("maximum out ", graph_stats_filtered_df["out"].max())

In [ ]:
# in-degree distribution (log-scaled y-axis)
in_stats_filtered_df = graph_stats_filtered_df.groupby(["in"]).count()
in_stats_filtered_df.rename({"out": "count"}, axis="columns", inplace=True)

print(in_stats_filtered_df.head(30))
plt.bar(in_stats_filtered_df.index, in_stats_filtered_df["count"])
plt.yscale("log")
plt.show()

In [ ]:
# out-degree distribution (log-scaled y-axis)
out_stats_filtered_df = graph_stats_filtered_df.groupby(["out"]).count()
out_stats_filtered_df.rename({"in": "count"}, axis="columns", inplace=True)

print(out_stats_filtered_df.head(30))
plt.bar(out_stats_filtered_df.index, out_stats_filtered_df["count"])
plt.yscale("log")
plt.show()

In [ ]:
# statistics about the connected components of the whole graph structure

# first, validate and clean the edge list

required_columns = {"txId1", "txId2"}

if not required_columns.issubset(edgelist_filtered_df.columns):
    missing = required_columns - set(edgelist_filtered_df.columns)
    raise ValueError(f"Missing columns: {missing}")

edges_filtered = (
    edgelist_filtered_df[["txId1", "txId2"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

print(f"Original number of edges: {len(edgelist_filtered_df):,}")
print(f"Number of cleaned unique edges: {len(edges_filtered):,}")

In [ ]:

# Second, create the directed graph, using networkx library

G_filtered = nx.from_pandas_edgelist(
    edges_filtered,
    source="txId1",
    target="txId2",
    create_using=nx.DiGraph(),
)

print(f"Number of nodes: {G_filtered.number_of_nodes():,}")
print(f"Number of edges: {G_filtered.number_of_edges():,}")

In [ ]:
# identify weakly connected components

components_filtered = list(nx.weakly_connected_components(G_filtered))

print(f"Number of connected components: {len(components_filtered):,}")

In [ ]:
# number of nodes per component
component_stats_filtered = pd.DataFrame(
    {
        "component_id": range(len(components_filtered)),
        "number_of_nodes": [len(component) for component in components_filtered],
    }
)

# number of edges per component
component_stats_filtered["number_of_edges"] = [
    G_filtered.subgraph(component).number_of_edges()
    for component in components_filtered
]

print(component_stats_filtered.head())

In [ ]:
# compute descriptive statistics

quantiles = [0.00, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]

node_size_statistics_filtered = (
    component_stats_filtered["number_of_nodes"]
    .describe(percentiles=quantiles[1:-1])
)

print("\nComponent size statistics, measured in nodes:")
print(node_size_statistics_filtered)

print("\nAdditional statistics:")
print(
    f"Mean: {component_stats_filtered['number_of_nodes'].mean():,.2f}"
)
print(
    f"Standard deviation: "
    f"{component_stats_filtered['number_of_nodes'].std():,.2f}"
)
print(
    f"Minimum: {component_stats_filtered['number_of_nodes'].min():,}"
)
print(
    f"Maximum: {component_stats_filtered['number_of_nodes'].max():,}"
)

print("\nQuantiles:")
print(
    component_stats_filtered["number_of_nodes"]
    .quantile(quantiles)
    .rename("number_of_nodes")
)

In [ ]:
plt.boxplot(component_stats_filtered["number_of_nodes"], showfliers=False)
plt.show()